# Sports Car Price Analysis
## Using Linear Regression, K-Means Clustering, and PCA
**Based on techniques from the Machine Learning and Data Mining course**

## 1. Data Loading and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy import linalg as LA

In [ ]:
# Load the sports car price dataset
data = pd.read_csv('Sport_car_price.csv')
print('Original dataset shape:', data.shape)
print('\nFirst 10 rows:')
print(data.head(10))

In [ ]:
# DISCLAIMER: Replace placeholder 9999 with NaN for electric vehicles
print('\n' + '='*80)
print('DATA PREPROCESSING DISCLAIMER')
print('='*80)
print('\n⚠️  This dataset includes 5 ELECTRIC VEHICLES:')
print('   - Rimac Nevera (2022)')
print('   - Rimac C_Two (2022)')
print('   - Tesla Roadster (2022)')
print('   - Tesla Model S Plaid (2021, 2022)')
print('   - Porsche Taycan (2021, 2022)')
print('\nThese vehicles do NOT have Internal Combustion Engines (ICE).')
print('Engine Size (L) feature is set to NaN for these cars.')
print('They will be EXCLUDED from Engine Size analysis only.')
print('\nAll models will use:')
print('   • 281 total cars in dataset')
print('   • 276 gas-powered vehicles for engine size analysis')
print('   • 281 cars for price/performance/clustering analysis')
print('='*80)

In [ ]:
# Replace 9999 with NaN for electric vehicles
data['Engine Size (L)'] = data['Engine Size (L)'].replace(9999, np.nan)

print('\nElectric vehicles marked with NaN in Engine Size:')
electric_vehicles = data[data['Engine Size (L)'].isnull()]
print(f'Count: {len(electric_vehicles)} electric vehicles')
print('\nList:')
print(electric_vehicles[['Car Make', 'Car Model', 'Year', 'Horsepower', '0-60 MPH Time (seconds)', 'Price (in USD)']])

In [ ]:
# Statistics for full dataset
print('\n=== FULL DATASET STATISTICS (281 cars) ===')
print(data.describe())

In [ ]:
# Statistics for gas-powered vehicles only (Engine Size is not NaN)
gas_vehicles = data[data['Engine Size (L)'].notna()]
print('\n=== GAS-POWERED VEHICLES STATISTICS (276 cars) ===')
print(f'Engine Size (L) - Mean: {gas_vehicles["Engine Size (L)"].mean():.2f}')
print(f'Engine Size (L) - Std Dev: {gas_vehicles["Engine Size (L)"].std():.2f}')
print(f'Engine Size (L) - Min: {gas_vehicles["Engine Size (L)"].min():.2f}')
print(f'Engine Size (L) - Max: {gas_vehicles["Engine Size (L)"].max():.2f}')
print(f'\nHorsepower - Mean: {data["Horsepower"].mean():.2f} (all {len(data)} cars)')
print(f'Torque - Mean: {data["Torque (lb-ft)"].mean():.2f} (all {len(data)} cars)')
print(f'Price - Mean: ${data["Price (in USD)"].mean():,.2f} (all {len(data)} cars)')

In [ ]:
# For modeling, create dataset without NaN engine sizes
data_clean = data.dropna(subset=['Engine Size (L)'])
print(f'\nDataset for modeling: {len(data_clean)} rows (gas-powered vehicles only)')

In [ ]:
# Extract features for modeling
features = ['Engine Size (L)', 'Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)']
X = data_clean[features].values
y = data_clean['Price (in USD)'].values

print(f'\n=== MODELING DATASET ===')
print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeature statistics (gas-powered vehicles):')
for i, feat in enumerate(features):
    print(f'{feat}: min={X[:, i].min():.2f}, max={X[:, i].max():.2f}, mean={X[:, i].mean():.2f}')

## 2. Linear Regression - Price Prediction
**Based on Class6LinearRegression.ipynb**

**Formula:** `Price = β₀ + β₁(Engine Size) + β₂(Horsepower) + β₃(Torque) + β₄(0-60 Time)`

**Dataset:** 276 gas-powered vehicles (electric vehicles excluded due to no ICE)

In [ ]:
# Linear Regression using least squares method
# Add bias term (column of ones)
n = X.shape[0]
A = np.vstack((np.ones(n), X.T)).T

print(f'Matrix A shape: {A.shape}')
print(f'Target y shape: {y.shape}')

In [ ]:
# Method: Using numpy.linalg.lstsq (Least Squares)
beta, rss, rank, s = LA.lstsq(A, y, rcond=None)

print('\n=== LINEAR REGRESSION COEFFICIENTS ===')
print(f'β₀ (Intercept): ${beta[0]:,.2f}')
for i, feat in enumerate(features):
    print(f'β{i+1} ({feat}): {beta[i+1]:,.2f}')
    
print(f'\nResidual Sum of Squares (RSS): ${rss[0]:,.2f}' if len(rss) > 0 else 'RSS: Not available')
print(f'Rank: {rank}')

In [ ]:
# Make predictions
y_pred = A.dot(beta)

# Calculate R² score
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

# Calculate RMSE and MAE
rmse = np.sqrt(np.mean((y - y_pred) ** 2))
mae = np.mean(np.abs(y - y_pred))

print('\n=== LINEAR REGRESSION PERFORMANCE ===')
print(f'R² Score: {r_squared:.4f}')
print(f'RMSE (Root Mean Squared Error): ${rmse:,.2f}')
print(f'MAE (Mean Absolute Error): ${mae:,.2f}')

In [ ]:
# Compare actual vs predicted prices
comparison = pd.DataFrame({
    'Actual Price': y,
    'Predicted Price': y_pred,
    'Error': y - y_pred,
    'Error %': ((y - y_pred) / y * 100)
})

print('\nFirst 10 predictions:')
print(comparison.head(10))
print(f'\nMean Prediction Error: ${comparison["Error"].mean():,.2f}')
print(f'Std Dev of Error: ${comparison["Error"].std():,.2f}')

In [ ]:
# Visualization: Actual vs Predicted
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y, y_pred, alpha=0.6, color='blue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Price (USD)')
plt.ylabel('Predicted Price (USD)')
plt.title('Linear Regression: Actual vs Predicted Price')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(y_pred, y - y_pred, alpha=0.6, color='green')
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Price (USD)')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residual Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. K-Means Clustering - Car Segmentation
**Based on Kmeans1.ipynb and Kmeans2.ipynb**

**Dataset:** All 281 vehicles (electric included, features used don't require ICE)

In [ ]:
from sklearn.cluster import KMeans

# Prepare data for clustering (using all cars, features that all have values)
clustering_features = ['Horsepower', 'Torque (lb-ft)', 'Price (in USD)']
X_cluster = data[clustering_features].values

print(f'Clustering dataset: {len(data)} cars (all vehicles, electric included)')
print(f'Clustering features: {clustering_features}')

In [ ]:
# K-Means with 3 clusters
kmeans_3 = KMeans(n_clusters=3, random_state=42).fit(X_cluster)
centroids_3 = kmeans_3.cluster_centers_
labels_3 = kmeans_3.labels_

print('\n=== K-MEANS CLUSTERING (k=3) ===')
print('Centroids (Horsepower, Torque, Price):')
for i, centroid in enumerate(centroids_3):
    print(f'Cluster {i}: HP={centroid[0]:.1f}, Torque={centroid[1]:.1f}, Price=${centroid[2]:,.0f}')

In [ ]:
# K-Means with 4 clusters
kmeans_4 = KMeans(n_clusters=4, random_state=42).fit(X_cluster)
centroids_4 = kmeans_4.cluster_centers_
labels_4 = kmeans_4.labels_

print('\n=== K-MEANS CLUSTERING (k=4) ===')
print('Centroids (Horsepower, Torque, Price):')
for i, centroid in enumerate(centroids_4):
    print(f'Cluster {i}: HP={centroid[0]:.1f}, Torque={centroid[1]:.1f}, Price=${centroid[2]:,.0f}')

In [ ]:
# Visualize 3 and 4 clusters: Horsepower vs Price
plt.figure(figsize=(14, 5))

# 3 Clusters
plt.subplot(1, 2, 1)
plt.scatter(X_cluster[:, 0], X_cluster[:, 2], c=labels_3.astype(float), s=70, alpha=0.6, cmap='viridis')
plt.scatter(centroids_3[:, 0], centroids_3[:, 2], c='red', s=200, marker='X', edgecolors='black', linewidth=2, label='Centroids')
plt.xlabel('Horsepower')
plt.ylabel('Price (USD)')
plt.title('K-Means Clustering (k=3): Horsepower vs Price')
plt.colorbar(label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)

# 4 Clusters
plt.subplot(1, 2, 2)
plt.scatter(X_cluster[:, 0], X_cluster[:, 2], c=labels_4.astype(float), s=70, alpha=0.6, cmap='plasma')
plt.scatter(centroids_4[:, 0], centroids_4[:, 2], c='red', s=200, marker='X', edgecolors='black', linewidth=2, label='Centroids')
plt.xlabel('Horsepower')
plt.ylabel('Price (USD)')
plt.title('K-Means Clustering (k=4): Horsepower vs Price')
plt.colorbar(label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize 3 and 4 clusters: Torque vs Price
plt.figure(figsize=(14, 5))

# 3 Clusters
plt.subplot(1, 2, 1)
plt.scatter(X_cluster[:, 1], X_cluster[:, 2], c=labels_3.astype(float), s=70, alpha=0.6, cmap='viridis')
plt.scatter(centroids_3[:, 1], centroids_3[:, 2], c='red', s=200, marker='X', edgecolors='black', linewidth=2, label='Centroids')
plt.xlabel('Torque (lb-ft)')
plt.ylabel('Price (USD)')
plt.title('K-Means Clustering (k=3): Torque vs Price')
plt.colorbar(label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)

# 4 Clusters
plt.subplot(1, 2, 2)
plt.scatter(X_cluster[:, 1], X_cluster[:, 2], c=labels_4.astype(float), s=70, alpha=0.6, cmap='plasma')
plt.scatter(centroids_4[:, 1], centroids_4[:, 2], c='red', s=200, marker='X', edgecolors='black', linewidth=2, label='Centroids')
plt.xlabel('Torque (lb-ft)')
plt.ylabel('Price (USD)')
plt.title('K-Means Clustering (k=4): Torque vs Price')
plt.colorbar(label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cluster analysis
print('\n=== CLUSTER DISTRIBUTION ===')
print('K-Means (k=3):')
for i in range(3):
    count = np.sum(labels_3 == i)
    print(f'  Cluster {i}: {count} cars ({count/len(labels_3)*100:.1f}%)')

print('\nK-Means (k=4):')
for i in range(4):
    count = np.sum(labels_4 == i)
    print(f'  Cluster {i}: {count} cars ({count/len(labels_4)*100:.1f}%)')

## 4. PCA - Principal Component Analysis
**Based on Class7PCA.ipynb**

**Dataset:** All 281 vehicles (electric included, features used don't require ICE)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Prepare data for PCA (use features that don't have NaN for electric vehicles)
pca_features = ['Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)', 'Price (in USD)']
X_pca = data[pca_features].values

print(f'PCA dataset: {len(data)} cars (all vehicles)')
print(f'Features: {pca_features}')
print('\nNote: Engine Size (L) excluded due to electric vehicles having no ICE')

In [ ]:
# Standardize the data (mean=0, std=1) - Like Class7PCA.ipynb
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)

print(f'\n=== DATA STANDARDIZATION (Like Class7PCA) ===')
print(f'Scaled data shape: {X_scaled.shape}')
print(f'Mean of scaled data: {np.mean(X_scaled):.10f} (should be ≈ 0)')
print(f'Std of scaled data: {np.std(X_scaled):.4f} (should be ≈ 1)')

In [ ]:
# Apply PCA with 2 components
pca_2 = PCA(n_components=2)
principal_components_2d = pca_2.fit_transform(X_scaled)

print(f'\n=== PCA WITH 2 COMPONENTS ===')
print(f'Principal components shape (2D): {principal_components_2d.shape}')
print(f'Explained variance ratio: {pca_2.explained_variance_ratio_}')
print(f'Total variance explained: {sum(pca_2.explained_variance_ratio_):.4f} ({sum(pca_2.explained_variance_ratio_)*100:.2f}%)')

In [ ]:
# Create DataFrame with principal components
pca_df_2d = pd.DataFrame(
    data=principal_components_2d,
    columns=['PC1', 'PC2']
)

print('\nPrincipal Component DataFrame (first 5 rows):')
print(pca_df_2d.head())

In [ ]:
# Visualize PCA 2D
plt.figure(figsize=(14, 5))

# Scatter plot colored by price
plt.subplot(1, 2, 1)
scatter = plt.scatter(principal_components_2d[:, 0], principal_components_2d[:, 1], 
                     c=data['Price (in USD)'].values, cmap='viridis', s=70, alpha=0.6)
plt.xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]*100:.2f}% variance)')
plt.ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]*100:.2f}% variance)')
plt.title('PCA: Sports Cars in 2D Space (Colored by Price)')
plt.colorbar(scatter, label='Price (USD)')
plt.grid(True, alpha=0.3)

# Scatter plot colored by horsepower
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(principal_components_2d[:, 0], principal_components_2d[:, 1], 
                      c=data['Horsepower'].values, cmap='plasma', s=70, alpha=0.6)
plt.xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]*100:.2f}% variance)')
plt.ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]*100:.2f}% variance)')
plt.title('PCA: Sports Cars in 2D Space (Colored by Horsepower)')
plt.colorbar(scatter2, label='Horsepower')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Apply PCA with 3 components to see cumulative variance
pca_3 = PCA(n_components=3)
principal_components_3d = pca_3.fit_transform(X_scaled)

print('\n=== PCA WITH 3 COMPONENTS ===')
print(f'Explained variance ratio: {pca_3.explained_variance_ratio_}')
print(f'Cumulative variance: {np.cumsum(pca_3.explained_variance_ratio_)}')
print(f'Total variance explained: {sum(pca_3.explained_variance_ratio_):.4f} ({sum(pca_3.explained_variance_ratio_)*100:.2f}%)')

In [ ]:
# Explained variance by component
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca_3.explained_variance_ratio_) + 1), pca_3.explained_variance_ratio_, alpha=0.7, color='blue')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Variance Explained by Each Component')
plt.xticks(range(1, 4))
plt.grid(True, alpha=0.3, axis='y')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(pca_3.explained_variance_ratio_) + 1), np.cumsum(pca_3.explained_variance_ratio_), 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Variance Explained')
plt.xticks(range(1, 4))
plt.ylim([0, 1.05])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Summary and Recommendations

In [ ]:
print('\n' + '='*80)
print('ANALYSIS SUMMARY - SPORTS CAR PRICE PREDICTION')
print('='*80)

print(f'\n📊 DATASET COMPOSITION:')
print(f'   • Total Cars: {len(data)}')
print(f'   • Gas-Powered Vehicles (ICE): {len(data_clean)}')
print(f'   • Electric Vehicles (no ICE): {len(electric_vehicles)}')

print('\n1. LINEAR REGRESSION RESULTS (Gas-Powered Vehicles Only):')
print(f'   ✓ R² Score: {r_squared:.4f}')
print(f'   ✓ RMSE: ${rmse:,.2f}')
print(f'   ✓ MAE: ${mae:,.2f}')
print(f'   ✓ Model explains {r_squared*100:.2f}% of price variance')
print(f'\n   Top 3 Impact Factors on Price:')
coeff_pairs = [(features[i], beta[i+1]) for i in range(len(features))]
coeff_pairs.sort(key=lambda x: abs(x[1]), reverse=True)
for idx, (feat, coeff) in enumerate(coeff_pairs[:3], 1):
    print(f'   {idx}. {feat}: ${coeff:,.2f} per unit')

print('\n2. K-MEANS CLUSTERING RESULTS (All 281 Vehicles):')
print(f'   ✓ 3 clusters identified distinct car market segments')
print(f'   ✓ 4 clusters provide finer granularity')
print(f'   ✓ Cars naturally group by: Performance level (HP/Torque) and Price tier')

print('\n3. PCA (DIMENSION REDUCTION) RESULTS (All 281 Vehicles):')
print(f'   ✓ 2 components explain {sum(pca_2.explained_variance_ratio_)*100:.2f}% of variance')
print(f'   ✓ 3 components explain {sum(pca_3.explained_variance_ratio_)*100:.2f}% of variance')
print(f'   ✓ Successfully reduced dimensions while preserving information')
print(f'   ✓ Note: Engine Size excluded for all vehicles (electric do not have ICE)')

print('\n4. BEST MODELS FOR THIS DATA:')
print('   ✓ LINEAR REGRESSION - Price prediction (gas-powered vehicles)')
print('   ✓ K-MEANS CLUSTERING - Market segmentation (all vehicles)')
print('   ✓ PCA - Visualization and feature understanding (all vehicles)')

print('\n' + '='*80)